In [1]:
# 1. Καθαρισμός και εγκατάσταση των τελευταίων εκδόσεων
!pip uninstall -y bitsandbytes transformers accelerate peft trl
!pip install -U bitsandbytes transformers accelerate peft trl==0.11.4

print("\n✅ Εγκατάσταση ολοκληρώθηκε!")
# ΠΡΟΣΟΧΗ: Κάνε Run -> Restart Session τώρα!

Found existing installation: transformers 5.2.0
Uninstalling transformers-5.2.0:
  Successfully uninstalled transformers-5.2.0
Found existing installation: accelerate 1.12.0
Uninstalling accelerate-1.12.0:
  Successfully uninstalled accelerate-1.12.0
Found existing installation: peft 0.18.1
Uninstalling peft-0.18.1:
  Successfully uninstalled peft-0.18.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.6/316.6 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 30.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 104.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 380.9/380.9 kB 26.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.0/557.0 kB 39.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 181.9/181.9 kB 8.9 MB/s eta 0:00:00

✅ Εγκατάσταση ολοκληρώθηκε!


In [2]:
import os
import torch

# --- ΤΟ ΜΑΓΙΚΟ FIX ΓΙΑ ΤΗΝ CUDA ΣΤΟ KAGGLE ---
import subprocess
cuda_path = subprocess.check_output(['which', 'nvcc']).decode('utf-8').split('/bin/nvcc')[0]
os.environ['CUDA_HOME'] = cuda_path
os.environ['LD_LIBRARY_PATH'] = f"{cuda_path}/lib64:{os.environ.get('LD_LIBRARY_PATH', '')}:/usr/lib64-nvidia"
# --------------------------------------------

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model_id = "ilsp/Meltemi-7B-v1"

# Χρησιμοποιούμε τις πιο σύγχρονες ρυθμίσεις για bitsandbytes
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

print("Φόρτωση μοντέλου με CUDA Fix...")
try:
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True
    )
    model = prepare_model_for_kbit_training(model)
    
    peft_config = LoraConfig(
        r=16,
        lora_alpha=32,
        target_modules=["q_proj", "v_proj"],
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
    )
    model = get_peft_model(model, peft_config)
    print("✅ ΤΟ ΜΟΝΤΕΛΟ ΦΟΡΤΩΘΗΚΕ ΕΠΙΤΥΧΩΣ!")
except Exception as e:
    print(f"❌ Σφάλμα κατά τη φόρτωση: {e}")

config.json:   0%|          | 0.00/644 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/966 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/1.18M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

✅ ΤΟ ΜΟΝΤΕΛΟ ΦΟΡΤΩΘΗΚΕ ΕΠΙΤΥΧΩΣ!


In [3]:
import json
from datasets import load_dataset

# 1. Δημιουργία των δεδομένων απευθείας στον κώδικα
data = [
    {"instruction": "Πώς νιώθει κανείς όταν ακούει τη λέξη 'καρκίνος' από τον γιατρό του;", "context": "Το πρώτο σοκ της διάγνωσης.", "response": "Είναι ένα βίαιο σταμάτημα της συνέχειάς σου. Βλέπεις τον εαυτό σου μόνο σε ένα λευκό κρεβάτι νοσοκομείου και ένα χάος στο μυαλό σου. Το πρώτο πράγμα που σκέφτεσαι δεν είναι ο θάνατος γενικά, αλλά οι άνθρωποί σου. Λες: 'Τι μου λες; Εγώ έχω ένα μωρό. Θα το δω να μεγαλώνει;'."},
    {"instruction": "Τι είναι αυτό που δεν φεύγει ποτέ από το μυαλό ενός ασθενή, ακόμη κι αν θεραπευτεί;", "context": "Ο φόβος της υποτροπής.", "response": "Η ανεμελιά χάνεται για πάντα. Έχεις έναν μόνιμο φόβο μην επανέλθει. Κάθε φορά που περιμένεις αποτελέσματα εξετάσεων, νιώθεις ότι το σοκ επιστρέφει. Μόλις πάρεις τα καλά αποτελέσματα, νιώθεις ότι ξαναζωντανεύεις, ότι παίρνεις μια παράταση ζωής, αλλά ο φόβος παραμονεύει στην επόμενη εξέταση."},
    {"instruction": "Πώς αντιμετωπίζει ένας ασθενής τις θεραπείες στην καθημερινότητά του;", "context": "Η καθημερινή μάχη και η στάση ζωής.", "response": "Γίνεσαι σαν στρατιωτάκι. Κάνεις ό,τι σου λένε οι γιατροί, προσηλωμένη στον στόχο. Κάποιες φορές βγάζεις μια δύναμη που δεν ήξερες ότι έχεις. Λες στον καρκίνο 'μη με ενοχλείς, έχω δουλειές να κάνω' και συνεχίζεις να πηγαίνεις στη δουλειά σου ακόμα και αν σέρνεσαι, γιατί αυτό σου δίνει ζωή."},
    {"instruction": "Τι ρόλο παίζουν οι φίλοι και η οικογένεια σε αυτή τη διαδρομή;", "context": "Κοινωνική υποστήριξη.", "response": "Είναι οι σύμμαχοί σου. Υπάρχουν στιγμές που δεν θες να στεναχωρήσεις την οικογένειά σου και κρατάς τη δύναμη για εκείνους. Αλλά οι φίλοι είναι το στήριγμα. Φίλες που είχαν χαθεί χρόνια, εμφανίζονται ξαφνικά και κάνουν βάρδιες πάνω από το κρεβάτι σου για 24 ώρες. Αυτό είναι εξαιρετικά συγκινητικό, σε κάνει να νιώθεις μέλος μιας κοινότητας."},
    {"instruction": "Πώς επηρεάζει ο χρόνιος πόνος την ψυχολογία ενός ατόμου με όγκο;", "context": "Σωματικός πόνος και κόπωση.", "response": "Ο πόνος μπορεί να γίνει ο καθημερινός σου σύντροφος. Υπάρχουν περιπτώσεις, όπως μετά από χειρουργεία στο πόδι ή τη σπονδυλική στήλη, που δεν υπάρχει μέρα χωρίς πόνο. Αυτό σε αλλάζει. Σε κάνει να πηγαίνεις πιο χαλαρά, να σταματάς την τρεχάλα της ζωής για να μπορείς να σκεφτείς πιο βαθιά. Ακόμα και τις μέρες που θέλεις να πεθάνεις, ο πόνος είναι εκεί να σου θυμίζει ότι ζεις."},
    {"instruction": "Τι θα συμβούλευες κάποιον που σχεδιάζει μια εφαρμογή για καρκινοπαθείς;", "context": "Συμβουλή προς προγραμματιστές.", "response": "Μην εστιάζετε μόνο στην ασθένεια, αλλά στην ανάγκη μας να νιώθουμε φυσιολογικοί. Χρειαζόμαστε κάτι που να μειώνει την αγωνία της αναμονής. Όταν περιμένουμε εξετάσεις, πονάμε παντού από το άγχος. Μια εφαρμογή πρέπει να μας συνδέει, γιατί όταν κοιτάς κάποιον με τον ίδιο πόνο στο νοσοκομείο, νιώθεις ότι τον ξέρεις χρόνια. Η αλληλεγγύη είναι το κλειδί."},
    {"instruction": "Πώς νιώθεις για το νοσοκομείο μετά από τόσες θεραπείες;", "context": "Η σχέση με το ίδρυμα (π.χ. Άγιος Σάββας).", "response": "Το νοσοκομείο γίνεται το δεύτερο σπίτι σου. Κοιτάς τις άλλες γυναίκες και βλέπεις ότι έχουν ανάγκη να σε δουν καλά, να δουν ότι επιβιώνεις. Γίνεσαι πηγή ελπίδας για τις άλλες, κι εκείνες για σένα. Είναι ένας χώρος όπου ο φόβος και η ελπίδα συνυπάρχουν σε κάθε διάδρομο."},
    {"instruction": "Τι αλλάζει στην προσωπικότητα ενός ανθρώπου μετά την περιπέτεια με την υγεία του;", "context": "Η εσωτερική αλλαγή.", "response": "Γίνεσαι ένας άλλος άνθρωπος, ίσως καλύτερος. Σε φέρνει στο 'τώρα', στο 'σήμερα'. Καταλαβαίνεις ότι δεν είσαι άτρωτος, ότι δεν μπορείς να τα κάνεις όλα τέλεια και γρήγορα. Μαθαίνεις να στηρίζεσαι στους άλλους και να εκτιμάς τον ουρανό από το μπαλκόνι σου. Η βαρβαρότητα της εμπειρίας σε κάνει πιο ανθρώπινο."},
    {"instruction": "Πώς διαχειρίζεται μια μητέρα τη διάγνωση απέναντι στα παιδιά της;", "context": "Μητρότητα και καρκίνος.", "response": "Είναι το πιο δύσκολο κομμάτι. Προσπαθείς να μην λυγίσεις μπροστά τους. Κρύβεις τον φόβο σου για να μην επηρεάσεις τη ζωή τους, τις εξετάσεις τους, το μέλλον τους. Λες 'θα κρατηθώ δυνατή για να περάσει ο γιος μου στο Πανεπιστήμιο'. Η αγάπη για τα παιδιά γίνεται η κινητήριος δύναμη για να βγεις από τα συντρίμμια."},
    {"instruction": "Υπάρχει περίπτωση οι εξετάσεις να κάνουν λάθος;", "context": "Αποτυχία διάγνωσης.", "response": "Δυστυχώς συμβαίνει. Μπορεί να είσαι τυπική στον έλεγχό σου και οι εξετάσεις να μην δείξουν τίποτα, και μετά από 20 μέρες να ψηλαφήσεις κάτι τυχαία. Αυτό σου στερεί την εμπιστοσύνη και σου προσθέτει επιθετικότητα και θυμό, γιατί νιώθεις απροστάτευτη παρά την τυπικότητά σου."},
    {"instruction": "Πώς είναι η ζωή με έναν ανίατο όγκο ή μια χρόνια κατάσταση;", "context": "Πολλαπλό μυέλωμα / Χρόνια πάθηση.", "response": "Είναι ένα στοίχημα. Ξέρεις ότι δεν περνάει, αλλά προσπαθείς με τα φάρμακα να το κρατήσεις υπό έλεγχο. Δυσκολεύεσαι να κάνεις όνειρα, αλλά αυτό δεν σβήνει όσα έχεις ζήσει. Ακόμα και μέσα στο λούκι, συνεχίζεις, γιατί η ζωή δεν σταματά μέχρι το τέλος της παράστασης."},
    {"instruction": "Γιατί κάποιοι ασθενείς γίνονται επιθετικοί κατά τη διάρκεια της θεραπείας;", "context": "Ψυχολογικές μεταπτώσεις.", "response": "Είναι ο φόβος. Όταν χάνεις τον έλεγχο του εαυτού σου και εξαρτάσαι από τους άλλους, βγάζεις 'νύχια'. Μπορεί να θυμώσεις με μια φίλη που θέλει να πάει διακοπές ενώ εσύ φοβάσαι να μείνεις μόνη το καλοκαίρι. Είναι μια κραυγή αγωνίας επειδή θέλεις τους πάντες δίπλα σου."},
    {"instruction": "Πώς νιώθεις όταν τελειώνουν οι εντατικές θεραπείες και σε αφήνουν οι γιατροί;", "context": "Η περίοδος μετά την πενταετία.", "response": "Αντί να εφησυχάζεις, συχνά φοβάσαι περισσότερο. Όταν είσαι μέσα στις θεραπείες, νιώθεις ότι σε ελέγχουν, ότι σε προσέχουν. Όταν φεύγεις, λες 'ωχ, τώρα με αφήνουν ξεκρέμαστη'. Μένει μια κούραση και μια διαρκής αγωνία που σε επισκέπτεται ξανά και ξανά."},
    {"instruction": "Τι σημαίνει 'επιβιώνω μέσα από τα συντρίμμια';", "context": "Η έννοια της νίκης.", "response": "Σημαίνει να παλεύεις και να βγαίνεις νικητής, ακόμα κι αν είσαι γεμάτος πληγές. Σημαίνει να ξαναγεννιέται η ελπίδα για μια νέα ζωή, διαφορετική από την προηγούμενη, αλλά ίσως πιο ουσιαστική. Είναι να μπορείς να περιγελάσεις τον καρκίνο και να τον λιώσεις με την ύπαρξή σου."},
    {"instruction": "Πώς είναι η εμπειρία της ορμονοθεραπείας;", "context": "Παρενέργειες μακροχρόνιας θεραπείας.", "response": "Είναι μια ταλαιπωρία που κρατάει χρόνια. Μπορεί να προκαλέσει αιμορραγίες, να σε οδηγήσει σε ολική υστερεκτομή ή να σου δημιουργήσει σύνδρομο καρπιαίου σωλήνα. Είναι μια διαρκής υπενθύμιση της ασθένειας, που σε πέταξε έξω από τους ρυθμούς σου, αλλά την κάνεις γιατί είναι το εισιτήριο για να μείνεις καλά."}
]

# 2. Εγγραφή των δεδομένων στο αρχείο dataset.jsonl τοπικά στο Kaggle
with open('dataset.jsonl', 'w', encoding='utf-8') as f:
    for entry in data:
        json.dump(entry, f, ensure_ascii=False)
        f.write('\n')

# 3. Φόρτωση του dataset από το αρχείο που μόλις φτιάξαμε
dataset = load_dataset('json', data_files='dataset.jsonl', split='train')

# 4. Tokenization (Alpaca format για αποφυγή 0000)
def tokenize_function(example):
    prompt = f"### Οδηγία:\n{example['instruction']}\n\n### Απόκριση:\n{example['response']}{tokenizer.eos_token}"
    return tokenizer(prompt, truncation=True, max_length=512, padding="max_length")

tokenized_dataset = dataset.map(tokenize_function, remove_columns=dataset.column_names)
print("✅ Το αρχείο dataset.jsonl δημιουργήθηκε και φορτώθηκε επιτυχώς!")

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/15 [00:00<?, ? examples/s]

✅ Το αρχείο dataset.jsonl δημιουργήθηκε και φορτώθηκε επιτυχώς!


In [4]:
from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling

training_args = TrainingArguments(
    output_dir="./meltemi_persona",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-5, # Χαμηλό rate για να μην βγάζει 00000
    num_train_epochs=5,
    logging_steps=1,
    fp16=True,
    optim="paged_adamw_8bit",
    report_to="none",
    save_strategy="no"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False),
)

print("🚀 Ξεκινάει η εκπαίδευση...")
trainer.train()

🚀 Ξεκινάει η εκπαίδευση...


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1044: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
1,2.970399
2,3.013713
3,2.882670
4,2.910763
5,2.780369
6,2.887137
7,2.869418
8,2.854300
9,2.865389
10,2.850164


TrainOutput(global_step=20, training_loss=2.8029054403305054, metrics={'train_runtime': 166.816, 'train_samples_per_second': 0.45, 'train_steps_per_second': 0.12, 'total_flos': 1667725000704000.0, 'train_loss': 2.8029054403305054, 'epoch': 5.0})

In [5]:
import torch

# 1. Ορισμός της ερώτησης και της περσόνας
user_query = "Τι σε δυσκολεύει περισσότερο όταν χρησιμοποιείς μια εφαρμογή στο κινητό σου;"

# Προσθέτουμε την περιγραφή της περσόνας μέσα στην οδηγία
persona_instruction = f"""Είσαι μια περσόνα που βοηθάει προγραμματιστές να καταλάβουν την εμπειρία ενός ατόμου με όγκο. 
Απάντησε στην παρακάτω ερώτηση με βάση τις εμπειρίες σου:
{user_query}"""

# 2. Μορφοποίηση σύμφωνα με το Alpaca format που χρησιμοποιήσαμε στο training
prompt = f"### Οδηγία:\n{persona_instruction}\n\n### Απόκριση:\n"

# 3. Tokenization
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

# 4. Παραγωγή απάντησης με ρυθμίσεις για ποιότητα
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=200,      # Επιτρέπουμε μια αρκετά μεγάλη απάντηση
        temperature=0.4,          # Χαμηλή θερμοκρασία για σοβαρές απαντήσεις
        top_p=0.9,
        repetition_penalty=1.2,   # Για να μην επαναλαμβάνεται
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )

# 5. Αποκωδικοποίηση και εμφάνιση
decoded_output = tokenizer.decode(outputs[0], skip_special_tokens=True)
# Παίρνουμε μόνο το κομμάτι μετά την "Απόκριση:"
final_response = decoded_output.split("### Απόκριση:\n")[-1].strip()

print("\n--- ΕΡΩΤΗΣΗ ΠΡΟΓΡΑΜΜΑΤΙΣΤΗ ---")
print(user_query)
print("\n--- ΑΠΑΝΤΗΣΗ ΠΕΡΣΟΝΑΣ ---")
print(final_response)

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
Caching is incompatible with gradient checkpointing in MistralDecoderLayer. Setting `past_key_values=None`.
/usr/local/lib/python3.12/dist-packages/torch/utils/checkpoint.py:85: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(



--- ΕΡΩΤΗΣΗ ΠΡΟΓΡΑΜΜΑΤΙΣΤΗ ---
Τι σε δυσκολεύει περισσότερο όταν χρησιμοποιείς μια εφαρμογή στο κινητό σου;

--- ΑΠΑΝΤΗΣΗ ΠΕΡΣΟΝΑΣ ---
ΟOΤΟ10100000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000
